In [4]:
"""
raga_classification.py
-----------------------
Classifies ragas using PCD (Pitch Class Distribution) and PCDD (PCD
Derivative) features, reproducing the two-part evaluation described in your
quoted paper:

  1. 10-fold cross-validation accuracy (the "in-sample" success rate).
  2. "Unseen" evaluation -- testing on performances/artists that never
     appeared in training at all, which is a much harder, more realistic test.

WHAT YOU NEED
Your dataset is currently split into one CSV per raga (e.g. "śrī.csv"),
each with 10 rows (one per song/performance), and columns:
  filepath, artist, raga, tonic_hz, n_notes, pcd_0..pcd_11, pcdd_0..pcdd_143

Put all your per-raga CSVs into one folder (e.g. 10 files for 10 ragas) and
point DATA_DIR at it below.

HOW TO RUN
    On Kaggle: paste this whole file into one cell, edit DATA_DIR, run.
    Or as a script:  python raga_classification.py --data_dir /path/to/csvs
"""

import os
import glob
import argparse
import unicodedata
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# ============================================================
# CONFIG -- edit this to match your setup
# ============================================================
DATA_DIR = "/kaggle/input/datasets/mishbhaul/hcm-pcdpcdd-features"   # folder containing one CSV per raga
N_CV_FOLDS = 10
UNSEEN_TEST_SIZE = 0.2   # fraction of ARTISTS (not songs) held out per raga for the unseen test
RANDOM_STATE = 42
# ============================================================


# ===========================================================================
# 1. LOAD AND COMBINE ALL PER-RAGA CSVs INTO ONE DATAFRAME
# ===========================================================================
def load_all_ragas(data_dir: str) -> pd.DataFrame:
    csv_paths = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in {data_dir}")

    dfs = []
    for path in csv_paths:
        df = pd.read_csv(path)
        dfs.append(df)
        print(f"  loaded {os.path.basename(path)}: {len(df)} rows, raga(s)={df['raga'].unique().tolist()}")

    combined = pd.concat(dfs, ignore_index=True)
    print(f"\nCombined dataset: {len(combined)} rows, {combined['raga'].nunique()} raga classes")
    return combined


# ===========================================================================
# 1b. PREPROCESSING -- run this on the combined dataframe BEFORE anything else.
#     Small hand-built datasets like this one commonly have quiet issues that
#     silently hurt a 100-row dataset far more than they would a huge one, so
#     each check below is intentionally verbose about what it finds/fixes.
# ===========================================================================
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    print(f"\n{'='*60}\nPREPROCESSING\n{'='*60}")
    df = df.copy()
    feature_cols = get_feature_columns(df)

    # --- 1. Normalize text columns (raga, artist) ---
    # Two different unicode encodings can render as the same visible text
    # (e.g. accented letters composed vs. pre-composed) but compare as
    # different strings -- silently splitting one raga into two label
    # classes. NFC normalization + whitespace stripping prevents that.
    for col in ["raga", "artist"]:
        if col in df.columns:
            before = df[col].nunique()
            df[col] = df[col].astype(str).str.strip().apply(lambda s: unicodedata.normalize("NFC", s))
            after = df[col].nunique()
            if before != after:
                print(f"  [fixed] '{col}': unicode/whitespace normalization merged "
                      f"{before} -> {after} unique values (likely duplicate categories).")
            else:
                print(f"  [ok] '{col}': {after} unique values, no normalization needed.")

    # --- 2. Drop exact duplicate rows ---
    # e.g. the same song accidentally processed and appended twice.
    n_before = len(df)
    df = df.drop_duplicates()
    if len(df) < n_before:
        print(f"  [fixed] dropped {n_before - len(df)} exact duplicate row(s).")
    else:
        print(f"  [ok] no exact duplicate rows found.")

    # --- 3. Duplicate (artist, raga, filepath) combos -- same song counted twice ---
    if {"artist", "raga", "filepath"}.issubset(df.columns):
        dupe_mask = df.duplicated(subset=["artist", "raga", "filepath"], keep="first")
        if dupe_mask.any():
            print(f"  [fixed] dropped {dupe_mask.sum()} row(s) with a repeated "
                  f"(artist, raga, filepath) combination.")
            df = df[~dupe_mask]
        else:
            print(f"  [ok] no repeated (artist, raga, filepath) combinations.")

    # --- 4. Missing values in feature columns ---
    n_missing = df[feature_cols].isna().sum().sum()
    if n_missing > 0:
        rows_with_na = df[feature_cols].isna().any(axis=1).sum()
        print(f"  [fixed] found {n_missing} missing feature value(s) across "
              f"{rows_with_na} row(s) -- dropping those rows (too few rows here "
              f"to safely impute).")
        df = df.dropna(subset=feature_cols)
    else:
        print(f"  [ok] no missing values in feature columns.")

    # --- 5. Infinite values in feature columns ---
    # can happen from upstream divide-by-zero during PCD/PCDD computation
    inf_mask = np.isinf(df[feature_cols].values).any(axis=1)
    if inf_mask.any():
        print(f"  [fixed] found {inf_mask.sum()} row(s) with infinite feature "
              f"values -- dropping them.")
        df = df[~inf_mask]
    else:
        print(f"  [ok] no infinite values in feature columns.")

    # --- 6. Sanity check: do PCD values actually sum to ~1 per row? ---
    # PCD is a pitch-class *distribution*, so each row's 12 pcd_* values
    # should sum to roughly 1.0 (it's a probability distribution over pitch
    # classes). This doesn't auto-fix anything -- it's a flag that the
    # upstream feature-extraction step may have a bug if it fails.
    pcd_cols = [c for c in feature_cols if c.startswith("pcd_") and not c.startswith("pcdd_")]
    if pcd_cols:
        row_sums = df[pcd_cols].sum(axis=1)
        bad_rows = ((row_sums < 0.95) | (row_sums > 1.05)).sum()
        if bad_rows > 0:
            print(f"  [warning] {bad_rows} row(s) have PCD values that don't sum to "
                  f"~1.0 (min={row_sums.min():.3f}, max={row_sums.max():.3f}). "
                  f"This may indicate a bug upstream in feature extraction -- "
                  f"not auto-fixed, please review these rows manually.")
        else:
            print(f"  [ok] PCD values sum to ~1.0 in every row, as expected.")

    # --- 7. Zero-variance (constant) feature columns ---
    # a column that's identical across every row adds no classification
    # signal and can cause numerical issues when scaling -- safe to drop.
    variances = df[feature_cols].var()
    zero_var_cols = variances[variances == 0].index.tolist()
    if zero_var_cols:
        print(f"  [fixed] dropping {len(zero_var_cols)} zero-variance (constant) "
              f"feature column(s): {zero_var_cols[:5]}{'...' if len(zero_var_cols) > 5 else ''}")
        df = df.drop(columns=zero_var_cols)
    else:
        print(f"  [ok] no zero-variance feature columns.")

    # --- 8. Class balance check ---
    # doesn't fix anything -- just flags it, since an imbalanced raga set
    # will skew both CV and unseen accuracy the way we discussed earlier
    # with the Bangla emotion dataset.
    class_counts = df["raga"].value_counts()
    if class_counts.min() != class_counts.max():
        print(f"  [warning] class counts are NOT balanced: "
              f"min={class_counts.min()} ({class_counts.idxmin()}), "
              f"max={class_counts.max()} ({class_counts.idxmax()}).")
    else:
        print(f"  [ok] all {len(class_counts)} raga classes have exactly "
              f"{class_counts.iloc[0]} songs each -- balanced.")

    # --- 9. Artist diversity per raga (needed for the unseen-evaluation split) ---
    artist_counts = df.groupby("raga")["artist"].nunique()
    single_artist_ragas = artist_counts[artist_counts < 2].index.tolist()
    if single_artist_ragas:
        print(f"  [warning] these raga(s) have only 1 unique artist, so no "
              f"'unseen' test split is possible for them: {single_artist_ragas}")
    else:
        print(f"  [ok] every raga has at least 2 unique artists "
              f"(min={artist_counts.min()}) -- unseen split is possible for all classes.")

    print(f"\nAfter preprocessing: {len(df)} rows, {df['raga'].nunique()} classes, "
          f"{len(get_feature_columns(df))} feature columns.")
    return df


# ===========================================================================
# 2. SPLIT OUT FEATURE COLUMNS (pcd_* and pcdd_*) FROM METADATA COLUMNS
# ===========================================================================
def get_feature_columns(df: pd.DataFrame) -> list:
    # auto-detect every pcd_/pcdd_ column, regardless of exactly how many
    # there are -- this way the script doesn't break if your feature count
    # ever changes (e.g. more/fewer PCDD bins).
    return [c for c in df.columns if c.startswith("pcd_") or c.startswith("pcdd_")]


def prepare_data(df: pd.DataFrame):
    feature_cols = get_feature_columns(df)
    X = df[feature_cols].values
    y_raw = df["raga"].values
    groups = df["artist"].values   # used later for the artist-independent "unseen" split

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_raw)

    print(f"Feature matrix: {X.shape}  ({len(feature_cols)} features)")
    print(f"Classes ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}")

    return X, y, groups, label_encoder


# ===========================================================================
# 3. THE THREE MODELS
# ===========================================================================
def build_models():
    return {
        "SVM": SVC(kernel="rbf", C=10, gamma="scale", random_state=RANDOM_STATE),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.1,
            eval_metric="mlogloss", random_state=RANDOM_STATE,
        ),
    }


# ===========================================================================
# 4. PART 1: 10-FOLD CROSS-VALIDATION
# ===========================================================================
def run_cv_evaluation(X, y, n_folds=N_CV_FOLDS):
    """
    Standard 10-fold stratified CV -- reports the "success rate" the way your
    quoted paper describes it. Features are standardized (mean 0, std 1)
    inside each fold's training data only, via a Pipeline, so there's no
    leakage from test fold into the scaling.
    """
    from sklearn.pipeline import Pipeline

    print(f"\n{'='*60}\n10-FOLD CROSS-VALIDATION\n{'='*60}")

    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    for name, model in build_models().items():
        pipeline = Pipeline([("scaler", StandardScaler()), ("clf", model)])
        scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy")
        results[name] = scores
        print(f"{name:10s}: mean accuracy = {scores.mean()*100:.2f}%  "
              f"(std = {scores.std()*100:.2f}%)  folds = {np.round(scores*100, 1)}")

    return results


# ===========================================================================
# 5. PART 2: "UNSEEN" EVALUATION -- artist-independent train/test split
# ===========================================================================
def artist_independent_split(df: pd.DataFrame, test_size=UNSEEN_TEST_SIZE, random_state=RANDOM_STATE,
                              max_attempts=50):
    """
    Splits by ARTIST GLOBALLY, not per-raga -- every artist ends up entirely
    in train OR entirely in test, across ALL the ragas they performed. This
    guarantees zero leakage even when the same artist (e.g. a famous
    vocalist) appears across multiple different ragas in the dataset, which
    a naive per-raga split would miss.

    Because some famous artists span many ragas, a single random split can
    accidentally leave a raga with zero test songs (if all its artists
    happen to land in train). To reduce that risk without ever risking
    leakage, this tries several random seeds and keeps whichever one covers
    the most raga classes in the test set -- but every attempt is leakage-free
    by construction (GroupShuffleSplit never splits one group across both
    sides), so this search only affects *class coverage*, never correctness.
    """
    all_ragas = set(df["raga"].unique())
    best_split = None
    best_coverage = -1

    for attempt in range(max_attempts):
        seed = random_state + attempt
        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_idx, test_idx = next(gss.split(df, groups=df["artist"]))
        train_df = df.iloc[train_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)

        coverage = test_df["raga"].nunique()
        if coverage > best_coverage:
            best_coverage = coverage
            best_split = (train_df, test_df)
        if coverage == len(all_ragas):
            break  # every raga is represented in the test set -- good enough, stop searching

    train_df, test_df = best_split

    # this should be mathematically guaranteed by GroupShuffleSplit, but
    # verify it explicitly anyway -- if this ever fires, something is wrong
    # with how the split was constructed, not just bad luck.
    overlap = set(train_df["artist"]) & set(test_df["artist"])
    assert not overlap, f"Artist leakage between train/test: {overlap}"

    missing_ragas = all_ragas - set(test_df["raga"].unique())
    if missing_ragas:
        print(f"  [warning] {len(missing_ragas)} raga(s) have NO songs in the unseen "
              f"test set after {max_attempts} attempts (all their artists landed in "
              f"training): {sorted(missing_ragas)}")
    else:
        print(f"  [ok] every raga has at least one song in the unseen test set "
              f"(found after {attempt + 1} attempt(s)).")

    return train_df, test_df


def run_unseen_evaluation(df: pd.DataFrame, label_encoder: LabelEncoder):
    print(f"\n{'='*60}\nUNSEEN EVALUATION (held-out artists/performances)\n{'='*60}")

    train_df, test_df = artist_independent_split(df)
    feature_cols = get_feature_columns(df)

    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    y_train = label_encoder.transform(train_df["raga"].values)
    y_test = label_encoder.transform(test_df["raga"].values)

    print(f"Train: {len(X_train)} songs | Unseen test: {len(X_test)} songs "
          f"(from artists not in training)")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    results = {}
    for name, model in build_models().items():
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        results[name] = acc
        print(f"\n{name}: unseen accuracy = {acc*100:.2f}%")
        print(classification_report(
            y_test, y_pred,
            labels=np.unique(np.concatenate([y_test, y_pred])),
            target_names=label_encoder.inverse_transform(np.unique(np.concatenate([y_test, y_pred]))),
            zero_division=0,
        ))

    return results


# ===========================================================================
# 6. ENTRY POINT
# ===========================================================================
def main(data_dir=DATA_DIR):
    print("Loading data...")
    df = load_all_ragas(data_dir)
    df = preprocess_data(df)

    X, y, groups, label_encoder = prepare_data(df)

    cv_results = run_cv_evaluation(X, y)
    unseen_results = run_unseen_evaluation(df, label_encoder)

    print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
    print(f"{'Model':10s} | {'10-fold CV':>12s} | {'Unseen':>10s}")
    print("-" * 38)
    for name in build_models().keys():
        cv_acc = cv_results[name].mean() * 100
        unseen_acc = unseen_results[name] * 100
        print(f"{name:10s} | {cv_acc:11.2f}% | {unseen_acc:9.2f}%")

    return cv_results, unseen_results

In [5]:
cv_results, unseen_results = main(data_dir="/kaggle/input/datasets/mishbhaul/hcm-pcdpcdd-features")

Loading data...
  loaded ahira_bhairav.csv: 10 rows, raga(s)=['Ahira bhairav']
  loaded bhairav.csv: 10 rows, raga(s)=['Bhairav']
  loaded bhg.csv: 10 rows, raga(s)=['Ābhōgī']
  loaded darbri.csv: 10 rows, raga(s)=['Darbāri']
  loaded kdr.csv: 10 rows, raga(s)=['Kēdār']
  loaded madhukauns.csv: 10 rows, raga(s)=['Madhukauns']
  loaded madhuvanti.csv: 10 rows, raga(s)=['Madhuvanti']
  loaded mrv.csv: 10 rows, raga(s)=['Mārvā']
  loaded r.csv: 10 rows, raga(s)=['Śrī']
  loaded rgr.csv: 10 rows, raga(s)=['Rāgēśrī']

Combined dataset: 100 rows, 10 raga classes

PREPROCESSING
  [ok] 'raga': 10 unique values, no normalization needed.
  [ok] 'artist': 38 unique values, no normalization needed.
  [ok] no exact duplicate rows found.
  [ok] no repeated (artist, raga, filepath) combinations.
  [ok] no missing values in feature columns.
  [ok] no infinite values in feature columns.
  [ok] PCD values sum to ~1.0 in every row, as expected.
  [ok] no zero-variance feature columns.
  [ok] all 10 raga 